# A2 — Knowledge-Base Demo (fill this)
Show OCR quality on a sample and one working retrieval example.

## Step 16 scratch — baseline OCR numbers (pretrained, not fine-tuned)

Full-book run on Kaggle GPU T4×2 (`facebook/nougat-base`, **not** fine-tuned —
fine-tuning is Sprint 4 / Step 28). `KAGGLE/kaggle.ipynb` produced `data/ocr/` locally; this cell reads it directly, so every number below is reproducible from
this repo's own state, not copied by hand.

This is the **BEFORE** number Step 29 will compare the fine-tuned reader against.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, "../src")  # notebook runs from notebooks/; doc_agent lives in ../src
from doc_agent.eval import metrics

OCR_DIR = Path("../data/ocr")
LABELS_PATH = Path("../grading_kit/labels.jsonl")
N_CONTENT_PAGES = 1040  # ingest/loader.py: as_p* pages after dropping blanks/front matter

failures = json.loads((OCR_DIR / "failures.json").read_text(encoding="utf-8"))
mmd_files = sorted(OCR_DIR.glob("*.mmd"))
by_reason: dict[str, int] = {}
for row in failures:
    by_reason[row["reason"]] = by_reason.get(row["reason"], 0) + 1

print(f"pages attempted      : {N_CONTENT_PAGES}")
print(f"transcripts produced : {len(mmd_files)}")
print(f"failed / degenerate  : {len(failures)}  ({100 * len(failures) / N_CONTENT_PAGES:.1f}%)")
for reason, count in sorted(by_reason.items(), key=lambda kv: -kv[1]):
    print(f"   {reason:<28} {count:>4}  ({100 * count / len(failures):.1f}% of failures)")

words = sum(len(p.read_text(encoding="utf-8").split()) for p in mmd_files)
print(f"\nwords from OUR OCR   : {words}  (task.yaml floor: 60,000 -- {'MET' if words >= 60000 else 'NOT MET'})")

gold: dict[str, str] = {}
for line in LABELS_PATH.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line:
        row = json.loads(line)
        gold[row["page_id"]] = row["text"]

print(f"\n{'page':<10} {'char-F1':>9} {'exact-form':>12} {'gold-form':>10} {'pred-form':>10}  status")
print("-" * 72)
failed_reason = {row["page_id"]: row["reason"] for row in failures}
scored = []
for pid in ("as_p0243", "as_p0255", "as_p0360"):
    mmd = OCR_DIR / f"{pid}.mmd"
    if not mmd.exists():
        reason = failed_reason.get(pid, "?")
        print(f"{pid:<10} {'--':>9} {'--':>12} {'--':>10} {'--':>10}  FAILED: {reason}")
        continue
    pred = mmd.read_text(encoding="utf-8")
    g = gold[pid]
    f1 = metrics.ocr_f1(pred, g)
    ex = metrics.exact_formula_match(pred, g)
    ngf = len(metrics.extract_formulas(g))
    npf = len(metrics.extract_formulas(pred))
    print(f"{pid:<10} {f1:>9.4f} {ex:>12.4f} {ngf:>10} {npf:>10}  ok")
    scored.append((pid, f1, ex, ngf))
print("-" * 72)

if scored:
    mean_f1 = sum(s[1] for s in scored) / len(scored)
    total_f = sum(s[3] for s in scored)
    weighted_ex = (
        sum(s[2] * s[3] for s in scored) / total_f if total_f else 0.0
    )
    print(f"mean char-F1 (n={len(scored)} gold page(s) with a transcript): {mean_f1:.4f}")
    print(f"exact-formula-match, weighted by formula count: {weighted_ex:.4f}")

print("\nWe expected  the FINE-TUNED reader to land at char-F1 0.88-0.93,")
print("exact-match 0.55-0.75. This is the pretrained BEFORE number Step 29 compares against.")


pages attempted      : 1040
transcripts produced : 594
failed / degenerate  : 446  (42.9%)
   empty-or-near-empty           314  (70.4% of failures)
   nougat-missing-page-marker     82  (18.4% of failures)
   repetition-degeneration        50  (11.2% of failures)

words from OUR OCR   : 87523  (task.yaml floor: 60,000 -- MET)

page         char-F1   exact-form  gold-form  pred-form  status
------------------------------------------------------------------------
as_p0243          --           --         --         --  FAILED: nougat-missing-page-marker
as_p0255          --           --         --         --  FAILED: repetition-degeneration
as_p0360      0.4166       0.0000         13          5  ok
------------------------------------------------------------------------
mean char-F1 (n=1 gold page(s) with a transcript): 0.4166
exact-formula-match, weighted by formula count: 0.0000

We expected  the FINE-TUNED reader to land at char-F1 0.88-0.93,
exact-match 0.55-0.75. This is the pretr

In [2]:
# IMPLEMENT: run OCR quality + one retrieval, end to end
